# 05 — Provider Master Build

## Objective
Build the provider analytical base:

`provider_master.csv`

This table consolidates:
- provider attributes
- provider-month features
- aggregated claim behavior

It will support:
- anomaly detection
- fraud exposure monitoring
- provider cost profiling

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth


In [2]:
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR
from src.data.load_data import (
    load_providers,
    load_provider_month_features,
    load_claims,
)

In [3]:
## Load source tables
providers = load_providers()
provider_month = load_provider_month_features()
claims = load_claims()

print("providers:", providers.shape)
print("provider_month:", provider_month.shape)
print("claims:", claims.shape)
display(providers.head(3))
display(provider_month.head(3))

providers: (180, 17)
provider_month: (4263, 24)
claims: (44144, 36)


,provider_id,provider_name,provider_type,specialty_group,region,city_tier,network_status,provider_archetype,contract_type,base_cost_multiplier,diagnostic_intensity,admission_intensity,average_claim_expected,historical_volume_band,historical_suspicion_flag,provider_quality_proxy,fraud_exposure_score
0,PRV00001,Provider 1,Clinic,Pediatrics,Central,Metro,In-network,Normal,External,1.003462,Medium,Medium,209.30,High,0,High,0.177
1,PRV00002,Provider 2,Specialist Center,Pulmonology,Asunción,Semi-Urban,In-network,Normal,Standard,1.082438,Medium,Low,258.15,Low,0,Medium,0.238
2,PRV00003,Provider 3,Pharmacy,Pharmacy,Asunción,Urban,In-network,High Complexity,Standard,1.577247,Medium,Medium,319.41,Low,0,Medium,0.252


,provider_id,year_month,claims_n,members_unique_n,billed_total,approved_total,avg_claim_amount,median_claim_amount,p95_claim_amount,emergency_claims_n,...,dx_proc_mismatch_n,high_severity_share,suspicious_claim_ratio,provider_type,specialty_group,provider_archetype,cost_vs_peer_index,utilization_density_index,anomaly_score_provider,provider_alert_level
0,PRV00001,2024-01,13,13,2742.69,2392.39,210.976154,146.86,500.6120,1,...,0,0.077,0.000,Clinic,Pediatrics,Normal,0.465,1.0,0.010,Normal
1,PRV00001,2024-02,15,15,4396.21,4124.88,293.080667,214.67,753.0730,0,...,0,0.067,0.067,Clinic,Pediatrics,Normal,0.712,1.0,0.033,Normal
2,PRV00001,2024-03,22,22,4743.86,4029.39,215.630000,174.76,432.4795,3,...,0,0.136,0.091,Clinic,Pediatrics,Normal,0.460,1.0,0.042,Normal


In [4]:
## Inspect provider keys and columns
print(providers.columns.tolist())
print(provider_month.columns.tolist())

['provider_id', 'provider_name', 'provider_type', 'specialty_group', 'region', 'city_tier', 'network_status', 'provider_archetype', 'contract_type', 'base_cost_multiplier', 'diagnostic_intensity', 'admission_intensity', 'average_claim_expected', 'historical_volume_band', 'historical_suspicion_flag', 'provider_quality_proxy', 'fraud_exposure_score']
['provider_id', 'year_month', 'claims_n', 'members_unique_n', 'billed_total', 'approved_total', 'avg_claim_amount', 'median_claim_amount', 'p95_claim_amount', 'emergency_claims_n', 'inpatient_claims_n', 'preventive_claims_n', 'duplicate_claims_n', 'suspicious_claims_n', 'dx_proc_mismatch_n', 'high_severity_share', 'suspicious_claim_ratio', 'provider_type', 'specialty_group', 'provider_archetype', 'cost_vs_peer_index', 'utilization_density_index', 'anomaly_score_provider', 'provider_alert_level']


In [5]:
## Aggregate claim behavior by provider
print(claims.columns.tolist())

['claim_id', 'member_id', 'policy_id', 'provider_id', 'claim_date', 'claim_year', 'claim_month', 'service_type', 'service_category', 'diagnosis_group', 'diagnosis_severity', 'procedure_code_group', 'procedure_complexity', 'emergency_flag', 'inpatient_flag', 'surgery_flag', 'pharmacy_flag', 'preventive_flag', 'chronic_related_flag', 'maternity_related_flag', 'claim_amount_billed', 'claim_amount_approved', 'claim_amount_rejected', 'out_of_pocket_estimated', 'claim_processing_days', 'approval_rate', 'duplicate_claim_flag', 'temporal_anomaly_flag', 'upcoding_suspected_flag', 'provider_inflation_flag', 'mismatch_dx_proc_flag', 'suspicious_claim_flag', 'fraud_pattern_type', 'member_archetype_snapshot', 'cost_expected_band', 'claim_outlier_score_latent']


In [6]:
provider_claims_agg = None

if "provider_id" in claims.columns:
    cost_col = None
    for candidate in ["claim_amount", "paid_amount", "approved_amount", "amount_paid", "cost_amount", "total_cost"]:
        if candidate in claims.columns:
            cost_col = candidate
            break

    agg_dict = {"provider_id": "size"}
    rename_map = {"provider_id": "claims_count"}

    if cost_col is not None:
        agg_dict[cost_col] = ["mean", "median", "sum"]

    provider_claims_agg = claims.groupby("provider_id").agg(agg_dict)

    if cost_col is not None:
        provider_claims_agg.columns = ["claims_count", "claim_cost_mean", "claim_cost_median", "claim_cost_sum"]
    else:
        provider_claims_agg.columns = ["claims_count"]

    provider_claims_agg = provider_claims_agg.reset_index()

provider_claims_agg.head(10) if provider_claims_agg is not None else print("No provider_id in claims")

,provider_id,claims_count
0,PRV00001,363
1,PRV00002,168
2,PRV00003,89
3,PRV00004,94
4,PRV00005,344
5,PRV00006,346
6,PRV00007,342
7,PRV00008,85
8,PRV00009,73
9,PRV00010,75


In [8]:
# Build provider master
provider_master = providers.copy()

if "provider_id" in provider_month.columns:
    # seleccionar solo columnas numéricas
    numeric_cols = provider_month.select_dtypes(include=["number"]).columns.tolist()

    # quitar provider_id si entró como numérica por algún motivo
    numeric_cols = [c for c in numeric_cols if c != "provider_id"]

    print("Numeric columns used from provider_month:")
    print(numeric_cols)

    if len(numeric_cols) > 0:
        provider_month_agg = (
            provider_month.groupby("provider_id")[numeric_cols]
            .mean()
            .reset_index()
        )

        provider_month_agg = provider_month_agg.rename(
            columns={col: f"{col}_mean" for col in numeric_cols}
        )

        provider_master = provider_master.merge(
            provider_month_agg,
            on="provider_id",
            how="left"
        )

if provider_claims_agg is not None:
    provider_master = provider_master.merge(
        provider_claims_agg,
        on="provider_id",
        how="left"
    )

print("provider_master:", provider_master.shape)
provider_master.head(3)

Numeric columns used from provider_month:
['claims_n', 'members_unique_n', 'billed_total', 'approved_total', 'avg_claim_amount', 'median_claim_amount', 'p95_claim_amount', 'emergency_claims_n', 'inpatient_claims_n', 'preventive_claims_n', 'duplicate_claims_n', 'suspicious_claims_n', 'dx_proc_mismatch_n', 'high_severity_share', 'suspicious_claim_ratio', 'cost_vs_peer_index', 'utilization_density_index', 'anomaly_score_provider']
provider_master: (180, 36)


,provider_id,provider_name,provider_type,specialty_group,region,city_tier,network_status,provider_archetype,contract_type,base_cost_multiplier,...,preventive_claims_n_mean,duplicate_claims_n_mean,suspicious_claims_n_mean,dx_proc_mismatch_n_mean,high_severity_share_mean,suspicious_claim_ratio_mean,cost_vs_peer_index_mean,utilization_density_index_mean,anomaly_score_provider_mean,claims_count
0,PRV00001,Provider 1,Clinic,Pediatrics,Central,Metro,In-network,Normal,External,1.003462,...,1.583333,0.291667,1.000000,0.0,0.068750,0.065875,0.769750,1.008375,0.050250,363
1,PRV00002,Provider 2,Specialist Center,Pulmonology,Asunción,Semi-Urban,In-network,Normal,Standard,1.082438,...,0.708333,0.125000,0.541667,0.0,0.091667,0.061792,0.862542,1.000000,0.055875,168
2,PRV00003,Provider 3,Pharmacy,Pharmacy,Asunción,Urban,In-network,High Complexity,Standard,1.577247,...,0.333333,0.125000,0.166667,0.0,0.046875,0.072917,1.010125,1.000000,0.088875,89


In [9]:
## Create first derived variables
if "claim_cost_mean" in provider_master.columns and "average_claim_expected" in provider_master.columns:
    provider_master["observed_vs_expected_claim_ratio"] = (
        provider_master["claim_cost_mean"] / provider_master["average_claim_expected"]
    ).replace([np.inf, -np.inf], np.nan)

if "claims_count" in provider_master.columns:
    provider_master["high_claim_volume_flag"] = (
        provider_master["claims_count"] >= provider_master["claims_count"].quantile(0.80)
    ).astype(int)

if "fraud_exposure_score" in provider_master.columns:
    provider_master["high_fraud_exposure_flag"] = (
        provider_master["fraud_exposure_score"] >= provider_master["fraud_exposure_score"].quantile(0.80)
    ).astype(int)

if "base_cost_multiplier" in provider_master.columns:
    provider_master["high_cost_provider_flag"] = (
        provider_master["base_cost_multiplier"] >= provider_master["base_cost_multiplier"].quantile(0.80)
    ).astype(int)

In [10]:
## Validation
validation = pd.DataFrame([
    {
        "check": "provider_id unique",
        "result": provider_master["provider_id"].nunique() == len(provider_master)
    },
    {
        "check": "rows preserved from providers",
        "result": len(provider_master) == len(providers)
    },
    {
        "check": "claims_count available",
        "result": "claims_count" in provider_master.columns
    }
])

validation

,check,result
0,provider_id unique,True
1,rows preserved from providers,True
2,claims_count available,True


In [11]:
## Quick provider profiling
display(
    provider_master["provider_type"]
    .value_counts(dropna=False)
    .rename_axis("provider_type")
    .reset_index(name="count")
)

display(
    provider_master["provider_archetype"]
    .value_counts(dropna=False)
    .rename_axis("provider_archetype")
    .reset_index(name="count")
)

,provider_type,count
0,Clinic,54
1,Hospital,36
2,Pharmacy,30
3,Lab,27
4,Specialist Center,17
5,Imaging,16


,provider_archetype,count
0,Normal,102
1,Expensive,43
2,High Complexity,22
3,Suspicious,13


In [12]:
profile_cols = [c for c in [
    "base_cost_multiplier",
    "diagnostic_intensity",
    "admission_intensity",
    "fraud_exposure_score",
    "claims_count",
    "claim_cost_mean",
    "claim_cost_sum",
    "observed_vs_expected_claim_ratio"
] if c in provider_master.columns]

provider_master[profile_cols].describe()

,base_cost_multiplier,fraud_exposure_score,claims_count
count,180.000000,180.000000,180.000000
mean,1.217640,0.280950,245.244444
std,0.282473,0.180704,144.188720
min,0.850000,0.120000,60.000000
25%,1.000581,0.183750,94.000000
50%,1.106078,0.226500,253.500000
75%,1.418722,0.300250,389.000000
max,2.314093,0.929000,444.000000


In [13]:
## Save provider master
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_DIR / "provider_master.csv"
provider_master.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\processed\provider_master.csv


In [14]:
## Final notes
notes = [
    "provider_master.csv is now ready for future provider anomaly analysis.",
    "This table already combines static provider profile with aggregated activity signals.",
    "Next step: start portfolio EDA using member_master and other processed bases."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

1. provider_master.csv is now ready for future provider anomaly analysis.
2. This table already combines static provider profile with aggregated activity signals.
3. Next step: start portfolio EDA using member_master and other processed bases.
